# Task 1: Prosthetic Performance Analysis
**Dataset:** Simulated EMG/usage dataset  
**Goal:** Analyze Accuracy vs User Comfort  
**Deliverable:** Insights Dashboard

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
print('Libraries loaded successfully.')

## 1. Simulate EMG / Usage Dataset

In [ ]:
np.random.seed(42)
n_samples = 200

user_ids = np.arange(1, n_samples + 1)
age = np.random.randint(18, 70, n_samples)
experience_months = np.random.randint(1, 60, n_samples)

# EMG signal features
emg_rms = np.random.uniform(0.1, 1.0, n_samples)          # Root Mean Square of EMG signal
emg_mav = np.random.uniform(0.05, 0.9, n_samples)         # Mean Absolute Value
signal_noise_ratio = np.random.uniform(5, 40, n_samples)  # SNR in dB

# Accuracy: higher EMG quality + more experience = better accuracy
accuracy = (
    0.4 * emg_rms +
    0.3 * (experience_months / 60) +
    0.2 * (signal_noise_ratio / 40) +
    0.1 * np.random.normal(0, 0.05, n_samples)
)
accuracy = np.clip(accuracy, 0, 1) * 100  # as percentage

# Comfort score: inverse relationship with high EMG intensity (muscle fatigue)
comfort_score = (
    10 - 4 * emg_rms +
    0.05 * experience_months / 60 * 4 +
    np.random.normal(0, 0.5, n_samples)
)
comfort_score = np.clip(comfort_score, 1, 10)  # scale 1-10

daily_usage_hours = np.random.uniform(1, 12, n_samples)
prosthetic_type = np.random.choice(['Myoelectric', 'Body-Powered', 'Hybrid'], n_samples)

df = pd.DataFrame({
    'user_id': user_ids,
    'age': age,
    'experience_months': experience_months,
    'prosthetic_type': prosthetic_type,
    'emg_rms': emg_rms.round(3),
    'emg_mav': emg_mav.round(3),
    'signal_noise_ratio_dB': signal_noise_ratio.round(2),
    'accuracy_pct': accuracy.round(2),
    'comfort_score': comfort_score.round(2),
    'daily_usage_hours': daily_usage_hours.round(2)
})

df.to_csv('emg_usage_dataset.csv', index=False)
print(f'Dataset shape: {df.shape}')
df.head()

## 2. Exploratory Data Analysis

In [ ]:
print('=== Dataset Summary ===')
print(df.describe().round(2))
print('\n=== Missing Values ===')
print(df.isnull().sum())

## 3. Insights Dashboard

In [ ]:
fig = plt.figure(figsize=(18, 14))
fig.suptitle('Prosthetic Performance Analysis — Insights Dashboard', fontsize=18, fontweight='bold', y=0.98)
gs = GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

# --- Plot 1: Accuracy vs Comfort Scatter ---
ax1 = fig.add_subplot(gs[0, :2])
colors = {'Myoelectric': '#2196F3', 'Body-Powered': '#FF9800', 'Hybrid': '#4CAF50'}
for ptype, grp in df.groupby('prosthetic_type'):
    ax1.scatter(grp['accuracy_pct'], grp['comfort_score'],
                label=ptype, alpha=0.7, color=colors[ptype], s=60)
ax1.set_xlabel('Accuracy (%)', fontsize=11)
ax1.set_ylabel('Comfort Score (1–10)', fontsize=11)
ax1.set_title('Accuracy vs User Comfort by Prosthetic Type', fontsize=12, fontweight='bold')
ax1.legend()
m, b = np.polyfit(df['accuracy_pct'], df['comfort_score'], 1)
x_line = np.linspace(df['accuracy_pct'].min(), df['accuracy_pct'].max(), 100)
ax1.plot(x_line, m * x_line + b, 'r--', linewidth=1.5, label='Trend')

# --- Plot 2: Distribution of Accuracy ---
ax2 = fig.add_subplot(gs[0, 2])
ax2.hist(df['accuracy_pct'], bins=20, color='#2196F3', edgecolor='white', alpha=0.85)
ax2.axvline(df['accuracy_pct'].mean(), color='red', linestyle='--', label=f"Mean: {df['accuracy_pct'].mean():.1f}%")
ax2.set_xlabel('Accuracy (%)')
ax2.set_ylabel('Count')
ax2.set_title('Accuracy Distribution', fontweight='bold')
ax2.legend(fontsize=9)

# --- Plot 3: Avg Accuracy by Prosthetic Type ---
ax3 = fig.add_subplot(gs[1, 0])
avg_acc = df.groupby('prosthetic_type')['accuracy_pct'].mean().sort_values()
bars = ax3.barh(avg_acc.index, avg_acc.values, color=[colors[t] for t in avg_acc.index])
ax3.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)
ax3.set_xlabel('Avg Accuracy (%)')
ax3.set_title('Avg Accuracy by Type', fontweight='bold')
ax3.set_xlim(0, 80)

# --- Plot 4: Avg Comfort by Prosthetic Type ---
ax4 = fig.add_subplot(gs[1, 1])
avg_comfort = df.groupby('prosthetic_type')['comfort_score'].mean().sort_values()
bars2 = ax4.barh(avg_comfort.index, avg_comfort.values, color=[colors[t] for t in avg_comfort.index])
ax4.bar_label(bars2, fmt='%.2f', padding=3, fontsize=9)
ax4.set_xlabel('Avg Comfort Score')
ax4.set_title('Avg Comfort by Type', fontweight='bold')
ax4.set_xlim(0, 10)

# --- Plot 5: Experience vs Accuracy ---
ax5 = fig.add_subplot(gs[1, 2])
ax5.scatter(df['experience_months'], df['accuracy_pct'], alpha=0.5, color='#9C27B0', s=30)
m2, b2 = np.polyfit(df['experience_months'], df['accuracy_pct'], 1)
x2 = np.linspace(0, 60, 100)
ax5.plot(x2, m2 * x2 + b2, 'r--', linewidth=1.5)
ax5.set_xlabel('Experience (months)')
ax5.set_ylabel('Accuracy (%)')
ax5.set_title('Experience vs Accuracy', fontweight='bold')

# --- Plot 6: EMG RMS vs Comfort ---
ax6 = fig.add_subplot(gs[2, 0])
ax6.scatter(df['emg_rms'], df['comfort_score'], alpha=0.5, color='#FF5722', s=30)
m3, b3 = np.polyfit(df['emg_rms'], df['comfort_score'], 1)
x3 = np.linspace(0, 1, 100)
ax6.plot(x3, m3 * x3 + b3, 'b--', linewidth=1.5)
ax6.set_xlabel('EMG RMS')
ax6.set_ylabel('Comfort Score')
ax6.set_title('EMG Intensity vs Comfort', fontweight='bold')

# --- Plot 7: Daily Usage Hours Distribution ---
ax7 = fig.add_subplot(gs[2, 1])
for ptype, grp in df.groupby('prosthetic_type'):
    ax7.hist(grp['daily_usage_hours'], bins=15, alpha=0.6, label=ptype, color=colors[ptype])
ax7.set_xlabel('Daily Usage (hours)')
ax7.set_ylabel('Count')
ax7.set_title('Daily Usage Distribution', fontweight='bold')
ax7.legend(fontsize=8)

# --- Plot 8: Correlation Heatmap ---
ax8 = fig.add_subplot(gs[2, 2])
corr = df[['accuracy_pct', 'comfort_score', 'emg_rms', 'experience_months', 'daily_usage_hours']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax8, linewidths=0.5, annot_kws={'size': 8})
ax8.set_title('Feature Correlation', fontweight='bold')
ax8.tick_params(axis='x', rotation=45, labelsize=7)
ax8.tick_params(axis='y', rotation=0, labelsize=7)

plt.savefig('prosthetic_insights_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Dashboard saved as prosthetic_insights_dashboard.png')

## 4. Key Insights

In [ ]:
print('=== KEY INSIGHTS ===')
print(f"\n1. Average Accuracy: {df['accuracy_pct'].mean():.1f}% | Std: {df['accuracy_pct'].std():.1f}%")
print(f"2. Average Comfort Score: {df['comfort_score'].mean():.2f} / 10")
corr_val = df['accuracy_pct'].corr(df['comfort_score'])
print(f"3. Accuracy-Comfort Correlation: {corr_val:.3f} ({'positive' if corr_val > 0 else 'negative'} relationship)")

best_type = df.groupby('prosthetic_type')['accuracy_pct'].mean().idxmax()
print(f"4. Best Accuracy Prosthetic Type: {best_type}")

most_comfortable = df.groupby('prosthetic_type')['comfort_score'].mean().idxmax()
print(f"5. Most Comfortable Prosthetic Type: {most_comfortable}")

exp_corr = df['experience_months'].corr(df['accuracy_pct'])
print(f"6. Experience vs Accuracy Correlation: {exp_corr:.3f}")
print("\nConclusion: More experienced users tend to achieve higher accuracy.")
print("High EMG signal intensity tends to reduce comfort (muscle fatigue).")